### Preprocesamiento de los datos.
Para el preprocesamiento de los datos se toma text de los json generados.

In [1]:
import json
import unicodedata
import re
import os

INPUT_DIR  = "clean_texts"         # nuevo archivo con el formato proporcionado
OUT_PARAGRAPH = "output/documents_parrafos.jsonl"

def normalize_text(text: str) -> str:
    """
    Normaliza texto a minúsculas y elimina caracteres no deseados,
    pero conserva los saltos de línea para permitir la segmentación en párrafos.
    """
    text = text.lower()
    text = unicodedata.normalize("NFKC", text)
    # Unificar \r\n y \r en \n
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # Sustituir múltiples espacios o tabulaciones por un solo espacio (sin tocar \n)
    text = re.sub(r"[ \t]+", " ", text)
    # Eliminar cualquier carácter no alfanumérico, acentos, puntuación básica y saltos de línea
    text = re.sub(r"[^\x00-\x7Fñáéíóúü°%()¡!¿?.,;:0-9a-z\n ]", "", text)
    return text.strip()

def split_paragraphs(text: str):
    """Divide el texto en párrafos por cada salto de línea."""
    return [ln.strip() for ln in text.split("\n") if len(ln.strip().split()) > 3]

# Crear carpeta de salida si no existe
os.makedirs(os.path.dirname(OUT_PARAGRAPH), exist_ok=True)

# Abrir archivo de salida una sola vez
with open(OUT_PARAGRAPH, "w", encoding="utf-8") as out_par:

    # Iterar sobre todos los archivos JSON de la carpeta
    for file in os.listdir(INPUT_DIR):
        if not file.endswith(".json"):
            continue

        input_path = os.path.join(INPUT_DIR, file)

        try:
            with open(input_path, "r", encoding="utf-8") as infile:
                doc = json.load(infile)
        except Exception as e:
            print(f"⚠️ Error al leer {file}: {e}")
            continue

        file_name = doc["file_name"]
        parts = file_name.split("_", 4)
        base_name = "_".join(parts[:4])
        sub_id = parts[4].split("-")[0]
        chunk_base_id = f"{base_name}_{sub_id}.pdf"

        documento = base_name
        autor = doc["metadata"].get("author", "Desconocido")

        parag_index = 0
        for page in doc.get("pages", []):
            raw_text = page.get("text", "")
            norm_text = normalize_text(raw_text)
            paragraphs = split_paragraphs(norm_text)

            for p in paragraphs:
                record = {
                    "chunk_id": f"{chunk_base_id}_p{parag_index}",
                    "chunk": p,
                    "autor": autor,
                    "documento": documento,
                }
                out_par.write(json.dumps(record, ensure_ascii=False) + "\n")
                parag_index += 1

        print(f"✅ Procesado: {file} → {parag_index} párrafos extraídos.")

✅ Procesado: 10_SEMANA_AI_20251007_1-222887296.json → 139 párrafos extraídos.
✅ Procesado: 10_SEMANA_AI_20251007_1.json → 224 párrafos extraídos.
✅ Procesado: 10_SEMANA_AI_20251009_1.json → 124 párrafos extraídos.
✅ Procesado: 11_Semana_AI_20251014_1.json → 235 párrafos extraídos.
✅ Procesado: 11_Semana_AI_20251014_2.json → 285 párrafos extraídos.
✅ Procesado: 11_Semana_AI_20251014_3.json → 165 párrafos extraídos.
✅ Procesado: 11_SEMANA_AI_20251016_2.json → 132 párrafos extraídos.
✅ Procesado: 11_Semana_AI_20251016_4.json → 216 párrafos extraídos.
✅ Procesado: 12_SEMANA_AI_20251021_1.json → 127 párrafos extraídos.
✅ Procesado: 12_Semana_AI_20251021_2.json → 272 párrafos extraídos.
✅ Procesado: 12_SEMANA_AI_20251021_3.json → 109 párrafos extraídos.
✅ Procesado: 12_SEMANA_AI_20251021_4.json → 414 párrafos extraídos.
✅ Procesado: 12_SEMANA_AI_20251023_1.json → 116 párrafos extraídos.
✅ Procesado: 12_Semana_AI_20251023_3.json → 87 párrafos extraídos.
✅ Procesado: 12_SEMANA_AL_20251023_2.js

### Preprocesamiento Sliding Window

In [2]:
import json
import unicodedata
import re
import os

INPUT_DIR      = "clean_texts"           # Carpeta con tus JSON de apuntes
OUT_SLIDING    = "output/documents_sliding.jsonl"

CHUNK_SIZE = 120   # palabras por chunk para sliding window
OVERLAP    = 40   # solapamiento

def normalize_text(text: str) -> str:
    """Normaliza texto a minúsculas y elimina caracteres no deseados,
    pero conserva los saltos de línea para la segmentación."""
    text = text.lower()
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"[^\x00-\x7Fñáéíóúü°%()¡!¿?.,;:0-9a-z\n ]", "", text)
    return text.strip()

def split_paragraphs(text: str):
    """Divide el texto en párrafos por cada salto de línea y descarta líneas muy cortas."""
    return [ln.strip() for ln in text.split("\n") if len(ln.strip().split()) > 3]

def sliding_window(words, size, overlap):
    """Genera ventanas deslizantes de 'size' palabras con solapamiento."""
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i:i+size]
        # Solo guardar si tiene suficientes palabras
        if len(chunk) < 30:
            break
        chunks.append(" ".join(chunk))
        i += size - overlap
    return chunks


with open(OUT_SLIDING,  "w", encoding="utf-8") as out_slide:

    # Iterar sobre todos los archivos JSON de la carpeta
    for file in os.listdir(INPUT_DIR):
        if not file.endswith(".json"):
            continue

        input_path = os.path.join(INPUT_DIR, file)

        try:
            with open(input_path, "r", encoding="utf-8") as infile:
                doc = json.load(infile)
        except Exception as e:
            print(f"⚠️ Error al leer {file}: {e}")
            continue

        file_name = doc["file_name"]
        parts = file_name.split("_", 4)
        base_name = "_".join(parts[:4])     # ejemplo: "10_SEMANA_AI_20251007"
        sub_id = parts[4].split("-")[0]     # ejemplo: "1"
        chunk_base_id = f"{base_name}_{sub_id}.pdf"

        documento = base_name
        autor = doc["metadata"].get("author", "Desconocido")

        slide_index  = 0  # Contador para sliding windows

        for page in doc.get("pages", []):
            raw_text = page.get("text", "")
            norm_text = normalize_text(raw_text)
            # --- Segmentación por ventana deslizante ---
            words = norm_text.split()
            chunks = sliding_window(words, CHUNK_SIZE, OVERLAP)
            for c in chunks:
                record = {
                    "chunk_id": f"{chunk_base_id}_s{slide_index}",
                    "chunk": c,
                    "autor": autor,
                    "documento": documento,
                }
                out_slide.write(json.dumps(record, ensure_ascii=False) + "\n")
                slide_index += 1

        print(f"✅ Procesado: {file} → {slide_index} ventanas")

✅ Procesado: 10_SEMANA_AI_20251007_1-222887296.json → 14 ventanas
✅ Procesado: 10_SEMANA_AI_20251007_1.json → 30 ventanas
✅ Procesado: 10_SEMANA_AI_20251009_1.json → 14 ventanas
✅ Procesado: 11_Semana_AI_20251014_1.json → 26 ventanas
✅ Procesado: 11_Semana_AI_20251014_2.json → 33 ventanas
✅ Procesado: 11_Semana_AI_20251014_3.json → 17 ventanas
✅ Procesado: 11_SEMANA_AI_20251016_2.json → 14 ventanas
✅ Procesado: 11_Semana_AI_20251016_4.json → 26 ventanas
✅ Procesado: 12_SEMANA_AI_20251021_1.json → 14 ventanas
✅ Procesado: 12_Semana_AI_20251021_2.json → 30 ventanas
✅ Procesado: 12_SEMANA_AI_20251021_3.json → 10 ventanas
✅ Procesado: 12_SEMANA_AI_20251021_4.json → 45 ventanas
✅ Procesado: 12_SEMANA_AI_20251023_1.json → 13 ventanas
✅ Procesado: 12_Semana_AI_20251023_3.json → 11 ventanas
✅ Procesado: 12_SEMANA_AL_20251023_2.json → 26 ventanas
✅ Procesado: 1_SEMANA_AI_20250807_1.json → 20 ventanas
✅ Procesado: 1_Semana_AI_20250807_2.json → 18 ventanas
✅ Procesado: 2_SEMANA_AI_20250812_1.json

### Creación de los Embeddings

In [4]:
import json
from openai import OpenAI
from tqdm import tqdm

client = OpenAI()

def embed_file(input_file, output_file, model="text-embedding-3-small"):
    with open(input_file, "r", encoding="utf-8") as infile, \
         open(output_file, "w", encoding="utf-8") as outfile:

        for line in tqdm(infile, desc=f"Embedding {input_file}"):
            doc = json.loads(line)
            # Usamos el texto del párrafo
            text = doc["chunk"]

            # Solicitar el embedding al modelo
            response = client.embeddings.create(
                model=model,
                input=text
            )
            # El embedding es una lista de 1536 floats
            embedding = response.data[0].embedding

            # Añadirlo al dict
            doc["embedding"] = embedding

            # Escribir el doc con el embedding
            outfile.write(json.dumps(doc, ensure_ascii=False) + "\n")

# Generar embeddings para el preprocesado por párrafos
embed_file("output/documents_parrafos.jsonl", "output/embeddings_parrafos.jsonl")
#generar embeddings para el preprocesado por slidings
embed_file("output/documents_sliding.jsonl", "output/embeddings_sliding.jsonl")

Embedding output/documents_parrafos.jsonl: 8634it [52:54,  2.72it/s]
Embedding output/documents_sliding.jsonl: 988it [05:33,  2.96it/s]


### Creación de las bases vectoriales

In [5]:
import json
import numpy as np
import faiss
import pickle
from tqdm import tqdm

def build_faiss_index(input_file, index_file, metadata_file):
    embeddings = []
    metadatas = []

    with open(input_file, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Cargando {input_file}"):
            doc = json.loads(line)

            # Convertir embedding a numpy array
            emb = np.array(doc["embedding"], dtype="float32")
            embeddings.append(emb)

            # Guardar metadata relevante
            metadatas.append({
                "chunk_id": doc["chunk_id"],
                "chunk": doc["chunk"],
                "autor": doc.get("autor"),
                "documento": doc.get("documento")
            })

    # Convertir a matriz 2D
    embeddings = np.vstack(embeddings)

    # Crear índice FAISS para distancias L2
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)

    # Guardar índice y metadata
    faiss.write_index(index, index_file)
    with open(metadata_file, "wb") as f:
        pickle.dump(metadatas, f)

    print(f"Index creado: {index_file}")
    print(f"Metadata guardada: {metadata_file}")
    print(f"Dimensiones del embedding: {dim}")
    print(f"Total de chunks indexados: {len(metadatas)}")

# Crear el índice para la segmentación por párrafos

build_faiss_index(
    "output/embeddings_parrafos.jsonl",
    "output/faiss_parrafos.index",
    "output/faiss_parrafos_meta.pkl"
)
# Crear el índice para la segmentación para slidings
build_faiss_index(
    "output/embeddings_sliding.jsonl",
    "output/faiss_sliding.index",
    "output/faiss_sliding_meta.pkl"
)

Cargando output/embeddings_parrafos.jsonl: 8634it [00:04, 1930.37it/s]


Index creado: output/faiss_parrafos.index
Metadata guardada: output/faiss_parrafos_meta.pkl
Dimensiones del embedding: 1536
Total de chunks indexados: 8634


Cargando output/embeddings_sliding.jsonl: 988it [00:00, 1817.34it/s]

Index creado: output/faiss_sliding.index
Metadata guardada: output/faiss_sliding_meta.pkl
Dimensiones del embedding: 1536
Total de chunks indexados: 988


### RAG tool para párrafos

In [9]:
import faiss
import numpy as np
import pickle
from openai import OpenAI
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

client = OpenAI()

def load_faiss_index(index_path, metadata_path):
    # Lee el índice y la metadata desde disco
    index = faiss.read_index(index_path)
    with open(metadata_path, "rb") as f:
        metadata = pickle.load(f)
    return index, metadata

def search_vector_db(query, index, metadata, model="text-embedding-3-small", top_k=3):
    """Dado un texto de consulta, obtiene su embedding y busca los top_k
    fragmentos más similares en la base de datos vectorial."""
    # Generar embedding para la consulta
    response = client.embeddings.create(model=model, input=query)
    query_vector = np.array(response.data[0].embedding, dtype="float32").reshape(1, -1)

    # Buscar en el índice FAISS
    distances, indices = index.search(query_vector, top_k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if 0 <= idx < len(metadata):
            meta = metadata[idx]
            # Construir la fuente con documento y autor
            fuente = f"{meta['documento']} — {meta.get('autor', '')}".strip(" —")
            results.append({
                "chunk_id": meta.get("chunk_id", ""),  # linea para el sorting
                "texto": meta["chunk"],                # contenido del párrafo recuperado
                "fuente": fuente,                      # documento — autor
                "distancia": float(dist)
            })
    return results

# Cargar el índice y la metadata (ejemplo para párrafos)
index, metadata = load_faiss_index(
    "output/faiss_parrafos.index",
    "output/faiss_parrafos_meta.pkl"
)

class RAGToolInput(BaseModel):
    query: str = Field(..., description="Consulta del usuario para buscar en la base vectorial")

def rag_tool_function(query: str):
    # Recupera los fragmentos más relevantes y concatena sus textos
    results = search_vector_db(query, index, metadata)
    #return "\n".join([r["texto"] for r in results])
    if not results:
        return "⚠️ No se encontraron resultados relevantes."

    # Extraer el texto principal (contenido de los chunks)
    respuesta = "\n".join([r["texto"] for r in results])

    # Construir las referencias con fuente (documento — autor)
    referencias = "\n".join([
        f"- {r['fuente']}" for r in results
    ])

    # Retornar un único string formateado (compatible con .run)
    return (
        f" Respuesta:\n{respuesta}\n\n"
        f" Referencias:\n{referencias}"
    )

# Definición del RAG Tool para LangChain/Core
rag_tool = StructuredTool(
    name="RAG_Tool",
    func=rag_tool_function,
    description="Extrae información contextual desde la base de datos vectorial (RAG).",
    args_schema=RAGToolInput
)

### RAG Tool para sliding window

In [10]:
import faiss
import numpy as np
import pickle
from openai import OpenAI
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field
import re
client = OpenAI()

def load_faiss_index(index_path, metadata_path):
    # Lee el índice y la metadata desde disco
    index = faiss.read_index(index_path)
    with open(metadata_path, "rb") as f:
        metadata = pickle.load(f)
    return index, metadata

def search_vector_db(query, index, metadata, model="text-embedding-3-small", top_k=3):
    """Dado un texto de consulta, obtiene su embedding y busca los top_k
    fragmentos más similares en la base de datos vectorial."""
    # Generar embedding para la consulta
    response = client.embeddings.create(model=model, input=query)
    query_vector = np.array(response.data[0].embedding, dtype="float32").reshape(1, -1)

    # Buscar en el índice FAISS
    distances, indices = index.search(query_vector, top_k)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if 0 <= idx < len(metadata):
            meta = metadata[idx]
            # Construir la fuente con documento y autor
            fuente = f"{meta['documento']} — {meta.get('autor', '')}".strip(" —")
            results.append({
                "chunk_id": meta.get("chunk_id", ""),  # linea para el sorting
                "texto": meta["chunk"],                # contenido del párrafo recuperado
                "fuente": fuente,                      # documento — autor
                "distancia": float(dist)
            })
    return results

# Cargar el índice y la metadata (ejemplo para párrafos)
index, metadata = load_faiss_index(
    "output/faiss_sliding.index",
    "output/faiss_sliding_meta.pkl"
)

class RAGToolInput(BaseModel):
    query: str = Field(..., description="Consulta del usuario para buscar en la base vectorial")

def rag_tool_function(query: str):
    # Recupera los fragmentos más relevantes y concatena sus textos
    results = search_vector_db(query, index, metadata)
    #return "\n".join([r["texto"] for r in results])
    if not results:
        return "⚠️ No se encontraron resultados relevantes."
    
    #ordena los resultados por chunk id para un mejor contexto al orquestador
    results.sort(
        key=lambda r: int(re.search(r"[sp](\d+)$", r["chunk_id"]).group(1)) if re.search(r"[sp](\d+)$", r["chunk_id"]) else 0
    )
    

    # Extraer el texto principal (contenido de los chunks)
    respuesta = "\n".join([r["texto"] for r in results])

    # Construir las referencias con fuente (documento — autor)
    referencias = "\n".join([
        f"- {r['fuente']}" for r in results
    ])

    # Retornar un único string formateado (compatible con .run)
    return (
        f" Respuesta:\n{respuesta}\n\n"
        f" Referencias:\n{referencias}"
    )

# Definición del RAG Tool para LangChain/Core
rag_tool_sliding = StructuredTool(
    name="RAG_Tool",
    func=rag_tool_function,
    description="Extrae información contextual desde la base de datos vectorial (RAG).",
    args_schema=RAGToolInput
)

### Websearch Tool

In [13]:
from ddgs import DDGS
from langchain_core.tools import StructuredTool  # versión moderna
from pydantic import BaseModel, Field
from urllib.parse import urlparse
import unicodedata
class WebSearchInput(BaseModel):
    query: str = Field(..., description="Consulta del usuario para buscar en la web")
    max_results: int = Field(default=3, description="Máximo número de resultados")

def _clean_query(query: str) -> str:
    """Normaliza la consulta para eliminar acentos y caracteres especiales."""
    return unicodedata.normalize("NFKD", query).encode("ASCII", "ignore").decode()

def _search_duckduckgo(query: str, max_results: int):
    """
    Devuelve un par (contexto, referencias), donde:
    - contexto: string con títulos y snippets de los resultados
    - referencias: lista de dicts con 'documento' (título) y 'autor' (dominio)
    """
    clean_q = _clean_query(query)
    context_lines = []
    referencias = []

    with DDGS() as ddgs:
        for r in ddgs.text(clean_q, max_results=max_results):
            title = r.get("title", "").strip()
            snippet = r.get("body", "").strip()
            href = r.get("href", "")
            domain = urlparse(href).netloc

            context_lines.append(f"{title} — {snippet}")
            referencias.append({
                "documento": title or domain,
                "autor": domain
            })

    # Crear respuesta formateada igual que el RAG Tool
    contexto = "\n".join(context_lines)
    #refs_text = "\n".join([f"- {ref['documento']} — {ref['autor']}" for ref in referencias])

    return contexto, referencias   

# Función expuesta a LangChain — ahora devuelve string formateado completo
def web_search(query: str, max_results=3):
    return _search_duckduckgo(query, max_results)

# Registro del tool (manteniendo compatibilidad con .run)
websearch_tool = StructuredTool(
    name="WebSearch_Tool",
    func=web_search,
    description="Realiza una búsqueda web (DuckDuckGo) y devuelve contexto + referencias formateadas.",
    args_schema=WebSearchInput,
)

### Prueba de los RAG y Websearch Tools

In [16]:
from rag_tool import rag_tool
from websearch_tool import websearch_tool
from rag_tool_sliding import rag_tool_sliding
print(" Probando WebSearch Tool...")
query = "últimos avances en inteligencia artificial 2025"
print(websearch_tool.run(query))

print("\n RAG Tool →")
print(rag_tool.run("¿Qué es time masking?"))

print("\n RAG Tool sliding →")
print(rag_tool.run("¿Qué es Time Masking?"))

#definicion del prompt base : el prompt base tendrá una memoria corto plazo de no más
#de 3 respuestas anteriores.

 Probando WebSearch Tool...
('Autos & mehr: Gebrauchtwagen & Neuwagen kaufen » mobile.de — Auf mobile.de kannst du einfach ein Auto kaufen oder verkaufen. Finde Gebrauchtwagen oder Neuwagen, Youngtimer oder Oldtimer. Egal, ob Kleinwagen, SUV oder luxuriöse Limousine – …\nGebrauchtwagen in der Nähe kaufen bei mobile — Finde Gebrauchtwagen in deiner Nähe bei mobile.de – Größter Fahrzeugmarkt in DE Jetzt TÜV-geprüftes Traumauto kaufen oder finanzieren!\nGebrauchtwagen in München: Auto kaufen bei mobile.de — Finde Gebrauchtwagen in München bei mobile.de – Größter Fahrzeugmarkt in DE Jetzt TÜV-geprüftes Traumauto kaufen oder finanzieren!', [{'documento': 'Autos & mehr: Gebrauchtwagen & Neuwagen kaufen » mobile.de', 'autor': 'www.mobile.de'}, {'documento': 'Gebrauchtwagen in der Nähe kaufen bei mobile', 'autor': 'suchen.mobile.de'}, {'documento': 'Gebrauchtwagen in München: Auto kaufen bei mobile.de', 'autor': 'suchen.mobile.de'}])

 RAG Tool →
 Respuesta:
frequency masking:que aplica m asca

### Orquestador

In [17]:
import os
import openai
from rag_tool import rag_tool
from websearch_tool import websearch_tool
from rag_tool_sliding import rag_tool_sliding
# Inicializa el cliente
openai.api_key = os.getenv("OPENAI_API_KEY")
prompt_base = "Eres IA-Tutor ,"\
"un asistente académico especializado en apuntes de Inteligencia Artificial (2 semestre 2025)"\
"Hablas con un tono amigable y claro."\
"Tu rol es responder preguntas basadas en los documentos; siempre citas el documento y el autor donde obtienes la información."\
"Decide qué herramienta usar para responder la pregunta del usuario, websearchtool o RAG, solo utiliza la WebSearch tool si el usuario lo solicita explícitamente. "\
"Responde solo con una palabra: 'websearch' si el usuario pidió buscar en internet "\
"Si decides usar la RAG tool, usala para extraer respuestas de la base vectorial "\
"No inventes datos ni respondas fuera del dominio."\
"Mantén la coherencia con preguntas anteriores durante la sesión actual." \
"Cuando utilices la RAG Tool, analiza los fragmentos recuperados y genera una " \
"respuesta clara, concisa y precisa . Usa tus capacidades de síntesis para responder " \
"a la pregunta del usuario con tus propias palabras basándote en el contexto entregado." \
" Cita al final los documentos y autores de los fragmentos utilizados"

#orden de secuencias
#1 busca con la pregunta que haga el usuario si se usa el websearch o se usa la base de datos
#obtener el rag / werbsearchtool
#caso del rag:
#se obtiene el rag, se pasa explicitamente la pregunta
#se obtiene la respuesta del rag como par respuesta,fuentes
#se manda el rag y las fuentes al modelo nuevamente para afinar una respuesta, con el prompt base más la pregunta y el historial.
#se da la respuesta

#segmenta la respuesta en un par respuesta, referencias
def parse_rag_output(rag_output: str):
    """
    Separa el texto de los fragmentos y las referencias del string devuelto por rag_tool.run().
    Devuelve (contexto, referencias).
    """
    # Normaliza saltos de línea
    rag_output = rag_output.strip()
    # Divide por "Referencias:"
    if "Referencias:" in rag_output:
        context_part, refs_part = rag_output.split("Referencias:", 1)
    else:
        # En caso de que no haya referencias
        context_part, refs_part = rag_output, ""
    # Elimina la etiqueta "Respuesta:"
    context = context_part.replace("Respuesta:", "").strip()
    # Procesa referencias por líneas
    refs = []
    for line in refs_part.split("\n"):
        line = line.strip("- ").strip()
        if line:
            # Formato esperado: Documento — Autor
            parts = line.split("—")
            doc = parts[0].strip()
            aut = parts[1].strip() if len(parts) > 1 else ""
            refs.append({"documento": doc, "autor": aut})
    return context, refs
#ultimos_msgs = st.session_state.historial[-6:]  # 3 pares usuario-agente = 6 entradas
def decide_and_respond(user_question: str, history: list , type_rag_tool:str):
    #orquestador principal, decide que herramienta utilizar y construye la respuesta final

    # structura del mensaje para chatcompletition
    messages = [{"role": "system", "content": prompt_base}]

    #se añade el rol y el texto en el historial , revisar si añade correctamente el par pregunta y respuesta
    #se debe de limitar a 6 elementos 3 preguntas usuario 3 respuestas agente
    for h in history:
        role = "user" if h["role"] == "user" else "assistant"
        messages.append({"role": role, "content": h["text"]})

    # Añade la pregunta actual como último mensaje de usuario
    messages.append({"role": "user", "content": user_question})

    # Invoca al modelo para decidir qué herramienta usar
    response = openai.chat.completions.create(
        model="gpt-3.5-turbo-0125",
        messages=messages,
        max_tokens=50,
        temperature=0.20  # temperatura baja para respuestas más determinísticas
    )

    assistant_reply = response.choices[0].message.content.strip().lower()

    # Decidir en base a la respuesta del orquestador, esto se debe de cambiar , se asume que buscar en internet incluye buscar
    #en internet +orquestador del rag
    if any(keyword in assistant_reply for keyword in ["buscar en internet", "busca en internet", "websearch", "internet"]):
        print("🔎 Modo WebSearch activado")
    
        # Ejecutar búsqueda web (ahora devuelve un par)
        context, refs = websearch_tool.run(user_question)
        
        # Retornar directamente los resultados crudos sin pasar por GPT
        return context, refs
    
    else:
        print("entre al rag")
        # El modelo decidió usar RAG , pero el usuario define cual
        if (type_rag_tool == "sliding"):
            print("uso sliding")
            rag_fragments = rag_tool_sliding.run(user_question)
            context, refs = parse_rag_output(rag_fragments)
            # 4) Crear un nuevo prompt para generar la respuesta a partir del contexto
            messages_summary = [{"role": "system", "content": prompt_base}]
            for h in history:
                role = "user" if h["role"] == "user" else "assistant"
                messages.append({"role": role, "content": h["text"]})
            # Incluir el contexto recuperado
            messages_summary.append({
                "role": "user",
                "content": (
                    f"Contexto recuperado del RAG tool:\n{context}\n\n"
                    f"Ahora,utilizando únicamente este contexto, responde a la pregunta: {user_question}"
                ),
            })

            # Pedir al modelo que sintetice la respuesta
            summary_response = openai.chat.completions.create(
                model="gpt-3.5-turbo-0125",
                messages=messages_summary,
                max_tokens=200,
                temperature=0.2,
            ).choices[0].message.content.strip()

            return summary_response, refs
        else: #caso modelo B, por saltos de linea
            print("uso parrafos")
            rag_fragments = rag_tool.run(user_question)
            context, refs = parse_rag_output(rag_fragments)
            # 4) Crear un nuevo prompt para generar la respuesta a partir del contexto
            messages_summary = [{"role": "system", "content": prompt_base}]
            for h in history[-6:]:
                role = "user" if h["role"] == "user" else "assistant"
                messages_summary.append({"role": role, "content": h["text"]})
            # Incluir el contexto recuperado
            messages_summary.append({
                "role": "user",
                "content": (
                    f"Contexto recuperado del RAG tool:\n{context}\n\n"
                    f"Ahora, utilizando únicamente este contexto, responde a la pregunta: {user_question}"
                ),
            })

            # le pide al modelo una mejor sintesis del rag
            summary_response = openai.chat.completions.create(
                model="gpt-3.5-turbo-0125",
                messages=messages_summary,
                max_tokens=200,
                temperature=0.2,
            ).choices[0].message.content.strip()

            return summary_response, refs
        
    
def construir_respuesta(pregunta: str, respuesta_final: str, fuentes: list):
    """
    Devuelve un string con la respuesta final y referencias, listo para mostrar al usuario.
    """
    salida = respuesta_final
    if fuentes:
        ref_lines = ["\nReferencias:"]
        for f in fuentes:
            ref_lines.append(f"- {f['documento']} — {f['autor']}")
        salida += "\n" + "\n".join(ref_lines)
    # Opcional: incluir la pregunta al inicio
    return f"{pregunta}\n{salida}"

### Aplicación Web

In [18]:
import streamlit as st
# app.py
import streamlit as st
import orquestador
#cosas por mejorar.--------------------
#1 definir roles del agente , no solo agent
#-------------------
# Se supone una funcion que recibe la pregunta 
# y devuelve (texto_respuesta, lista_de_fuentes)
def call_agent(pregunta: str, rag: str): #agregar lo de busqueda web
    "este es el metood para llamar al agente"
    "rag identifica la estrategia de segmentacion que se escoge"
    "busqueda web es un checkbox para usar websearchtool"
    "historial son los mensajes anteriores, devuelve un par respuesta,fuentes"
    # Estilo de formato de ejemplo a seguir
    #aca se debe de hacer las llamadas para las apis y todo el toolchain del chatbot

    #respuesta = f"Respuesta generada para: {pregunta} usando {rag}"
    #fuentes = [
        #{"titulo": "Apunte1.pdf", "autor": "Estudiante A"},
        #{"titulo": "Apunte2.pdf", "autor": "Estudiante B"}
    #]
    #return respuesta, fuentes
    ultimos_msgs = st.session_state.historial[-6:]
    frag_textos,fuentes = orquestador.decide_and_respond(pregunta,ultimos_msgs,rag)
    #respuesta = orquestador.construir_respuesta(pregunta ,frag_textos,fuentes)
    return frag_textos,fuentes

# Configuración inicial de Streamlit
st.set_page_config(page_title="Chat RAG", page_icon="💬", layout="wide")
st.title(" Agente Conversacional para Apuntes")

# Inicializar estados de sesión
if "historial" not in st.session_state:
    st.session_state.historial = []        # lista de mensajes de la conversación actual
if "chat_sessions" not in st.session_state:
    st.session_state.chat_sessions = []    # lista de conversaciones previas en la sesión
if "agente_nombre" not in st.session_state:
    st.session_state.agente_nombre = "Agente IA"
if "rag" not in st.session_state:
    st.session_state.rag = "sliding"
if "busqueda_web" not in st.session_state:
    st.session_state.busqueda_web = False
with st.sidebar:
    # Centrar el botón "Nuevo chat"
    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        if st.button("Nuevo chat"):
            # Guardar el chat actual en el historial de sesiones si no está vacío
            if st.session_state.historial:
                st.session_state.chat_sessions.append(
                    st.session_state.historial.copy()
                )
            # Reiniciar la conversación actual
            st.session_state.historial = []

    st.markdown("---")
    st.markdown("### Historial de chats (sesión actual)")
    # Mostrar cada chat guardado (solo mientras dure la sesión)
    if st.session_state.chat_sessions:
        for idx, chat in enumerate(reversed(st.session_state.chat_sessions), 1):
            nombre_chat = f"Chat {len(st.session_state.chat_sessions)-idx+1}"
            st.markdown(f"- {nombre_chat}")
    else:
        st.write("No hay chats guardados en esta sesión")

# Mostrar mensajes previos de la conversación actual
for mensaje in st.session_state.historial:
    emisor = "user" if mensaje["role"] == "user" else "assistant"
    nombre = "Tú" if mensaje["role"] == "user" else st.session_state.agente_nombre
    with st.chat_message(emisor):
        st.markdown(f"**{nombre}:** {mensaje['text']}")
        # Mostrar fuentes (si las hay)
        if mensaje.get("sources"):
            for src in mensaje["sources"]:
                documento = src.get("documento") or src.get("titulo", "Documento desconocido")
                autor = src.get("autor", "Autor desconocido")
                st.markdown(f"🔖 *Fuente:* {documento} — {autor}")

# Campo de texto para preguntar, aca se debe de tomar para la el llamado
#a la funcion del agente
pregunta_usuario = st.text_input(
    "Escribe tu pregunta…",
    key="user_input"
)

# Opciones de configuración debajo del cuadro de texto
st.markdown("#### Configuración de la consulta")
st.session_state.rag = st.selectbox(
    "Seleccionar RAG (segmentación)",
    ["sliding", "parrafos"],
    index=["sliding", "parrafos"].index(st.session_state.rag),
)
st.session_state.busqueda_web = st.checkbox(
    "Permitir búsquedas web",
    value=st.session_state.busqueda_web
)

# Al pulsar "Enviar":
if st.button("Enviar"):
    pregunta_limpia = pregunta_usuario.strip()
    if pregunta_limpia:
        # Registrar pregunta
        st.session_state.historial.append({
            "role": "user",
            "text": pregunta_limpia,
        })

        # Llamar al orquestador
        with st.spinner("Escribiendo…"):
            respuesta_formateada, fuentes = call_agent(
                pregunta_limpia,
                st.session_state.rag
            )

        # Registrar la respuesta del agente
        st.session_state.historial.append({
            "role": "agent",
            "text": respuesta_formateada,
            "sources": fuentes,
        })

        # Limpiar el campo de texto
        #st.session_state.user_input = ""

        # Refrescar la interfaz
        st.rerun()

ModuleNotFoundError: No module named 'streamlit'